In [4]:
import os
import time
import requests
import pandas as pd
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv

load_dotenv()


BASE_URL = "https://api.fda.gov/drug/event.json"


def call_openfda(
    search: Optional[str] = None,
    count: Optional[str] = None,
    limit: int = 10,
    skip: int = 0,
) -> Dict[str, Any]:
    """
    Calls the openFDA Drug Event API.

    Parameters:
        search: openFDA search query, for example:
                'patient.reaction.reactionmeddrapt:"headache"'
        count: field to count by, for example:
               'patient.reaction.reactionmeddrapt.exact'
        limit: number of records or count buckets.
        skip: pagination offset.

    Returns:
        JSON response as a Python dictionary.
    """

    params = {
        "limit": limit,
        "skip": skip,
    }

    api_key = os.getenv("OPENFDA_API_KEY")
    if api_key:
        params["api_key"] = api_key

    if search:
        params["search"] = search

    if count:
        params["count"] = count

    response = requests.get(BASE_URL, params=params, timeout=30)

    if response.status_code != 200:
        raise RuntimeError(
            f"API request failed.\n"
            f"Status code: {response.status_code}\n"
            f"URL: {response.url}\n"
            f"Response: {response.text[:500]}"
        )

    return response.json()


def safe_get(dictionary: Dict[str, Any], path: List[str], default=None):
    """
    Safely gets a nested value from a dictionary.
    """
    current = dictionary

    for key in path:
        if not isinstance(current, dict):
            return default
        current = current.get(key)

        if current is None:
            return default

    return current


def flatten_report(report: Dict[str, Any]) -> Dict[str, Any]:
    """
    Converts one nested adverse event report into one flat row.

    Important:
    A report may contain multiple drugs and multiple reactions.
    This flattening keeps them as joined text fields.
    """

    patient = report.get("patient", {})

    reactions = patient.get("reaction", [])
    drugs = patient.get("drug", [])

    reaction_terms = [
        reaction.get("reactionmeddrapt")
        for reaction in reactions
        if reaction.get("reactionmeddrapt")
    ]

    medicinal_products = [
        drug.get("medicinalproduct")
        for drug in drugs
        if drug.get("medicinalproduct")
    ]

    drug_roles = [
        drug.get("drugcharacterization")
        for drug in drugs
        if drug.get("drugcharacterization")
    ]

    return {
        "safetyreportid": report.get("safetyreportid"),
        "receivedate": report.get("receivedate"),
        "receiptdate": report.get("receiptdate"),
        "serious": report.get("serious"),
        "seriousnessdeath": report.get("seriousnessdeath"),
        "seriousnesshospitalization": report.get("seriousnesshospitalization"),
        "seriousnesslifethreatening": report.get("seriousnesslifethreatening"),
        "occurcountry": report.get("occurcountry"),
        "primarysourcecountry": report.get("primarysourcecountry"),
        "patient_age": patient.get("patientonsetage"),
        "patient_age_unit": patient.get("patientonsetageunit"),
        "patient_sex": patient.get("patientsex"),
        "reaction_terms": " | ".join(reaction_terms),
        "drug_names": " | ".join(medicinal_products),
        "drug_roles": " | ".join(drug_roles),
        "number_of_reactions": len(reaction_terms),
        "number_of_drugs": len(medicinal_products),
    }


def fetch_sample_by_reaction(reaction_term: str = "headache", limit: int = 25) -> pd.DataFrame:
    """
    Fetches a sample of adverse event reports by reaction term.
    """

    search_query = f'patient.reaction.reactionmeddrapt:"{reaction_term}"'

    data = call_openfda(
        search=search_query,
        limit=limit,
    )

    results = data.get("results", [])

    rows = [flatten_report(report) for report in results]

    df = pd.DataFrame(rows)

    return df


def fetch_counts(
    search: Optional[str],
    count_field: str,
    limit: int = 20,
) -> pd.DataFrame:
    """
    Fetches count aggregations from openFDA.
    """

    data = call_openfda(
        search=search,
        count=count_field,
        limit=limit,
    )

    results = data.get("results", [])

    return pd.DataFrame(results)


def add_year_column(df: pd.DataFrame, date_column: str = "receivedate") -> pd.DataFrame:
    """
    Adds a year column from an openFDA date field formatted as YYYYMMDD.
    """

    df = df.copy()

    if date_column in df.columns:
        df["year"] = pd.to_datetime(
            df[date_column],
            format="%Y%m%d",
            errors="coerce"
        ).dt.year

    return df


def main():
    reaction_term = "headache"
    sample_limit = 50

    print(f"Fetching sample reports for reaction: {reaction_term}")

    df = fetch_sample_by_reaction(
        reaction_term=reaction_term,
        limit=sample_limit,
    )

    df = add_year_column(df)

    print("\nSample shape:")
    print(df.shape)

    print("\nFirst rows:")
    print(df.head())

    output_file = f"openfda_drug_event_sample_{reaction_term}.csv"
    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"\nCSV exported: {output_file}")

    search_query = f'patient.reaction.reactionmeddrapt:"{reaction_term}"'

    print("\nTop reported reactions within this search:")
    reaction_counts = fetch_counts(
        search=search_query,
        count_field="patient.reaction.reactionmeddrapt.exact",
        limit=20,
    )
    print(reaction_counts)

    reaction_counts.to_csv(
        f"openfda_top_reactions_{reaction_term}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    time.sleep(1)

    print("\nTop reported medicinal products within this search:")
    drug_counts = fetch_counts(
        search=search_query,
        count_field="patient.drug.medicinalproduct.exact",
        limit=20,
    )
    print(drug_counts)

    drug_counts.to_csv(
        f"openfda_top_drugs_{reaction_term}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    time.sleep(1)

    print("\nCounts by primary source country:")
    country_counts = fetch_counts(
        search=search_query,
        count_field="primarysourcecountry.exact",
        limit=20,
    )
    print(country_counts)

    country_counts.to_csv(
        f"openfda_country_counts_{reaction_term}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print("\nLocal quick summary:")
    if not df.empty:
        print("\nSerious field counts:")
        print(df["serious"].value_counts(dropna=False))

        print("\nReports by year:")
        print(df["year"].value_counts(dropna=False).sort_index())

        print("\nAverage number of drugs per report:")
        print(df["number_of_drugs"].mean())

        print("\nAverage number of reactions per report:")
        print(df["number_of_reactions"].mean())

    print("\nImportant interpretation warning:")
    print(
        "These are adverse event reports, not confirmed causal events. "
        "A report may contain multiple drugs and multiple reactions. "
        "Counts should not be interpreted as incidence or proof of causality."
    )
    return df

if __name__ == "__main__":
    df = main()

Fetching sample reports for reaction: headache

Sample shape:
(50, 18)

First rows:
  safetyreportid receivedate receiptdate serious seriousnessdeath  \
0       10003300    20140306    20140306       1             None   
1       10003320    20140312    20150812       2             None   
2       10003327    20140312    20140312       1             None   
3       10003344    20140312    20150812       2             None   
4       10003375    20140312    20140312       1             None   

  seriousnesshospitalization seriousnesslifethreatening occurcountry  \
0                        NaN                        NaN          NaN   
1                        NaN                        NaN           US   
2                          1                        NaN           CN   
3                        NaN                        NaN           US   
4                          1                        NaN           CN   

  primarysourcecountry patient_age patient_age_unit patient_sex  \
0

In [10]:
print (df.columns)
df.head()

Index(['safetyreportid', 'receivedate', 'receiptdate', 'serious',
       'seriousnessdeath', 'seriousnesshospitalization',
       'seriousnesslifethreatening', 'occurcountry', 'primarysourcecountry',
       'patient_age', 'patient_age_unit', 'patient_sex', 'reaction_terms',
       'drug_names', 'drug_roles', 'number_of_reactions', 'number_of_drugs',
       'year'],
      dtype='str')


,safetyreportid,receivedate,receiptdate,serious,seriousnessdeath,seriousnesshospitalization,seriousnesslifethreatening,occurcountry,primarysourcecountry,patient_age,patient_age_unit,patient_sex,reaction_terms,drug_names,drug_roles,number_of_reactions,number_of_drugs,year
0,10003300,20140306,20140306,1,None,NaN,NaN,NaN,US,77,801,2,Vomiting | Diarrhoea | Arthralgia | Headache,BONIVA,1,4,1,2014
1,10003320,20140312,20150812,2,None,NaN,NaN,US,US,42,801,1,Headache | Hypotension,TYVASO | LOSARTAN. | LETAIRIS | LOSARTAN. | LE...,1 | 1 | 1 | 1 | 1,2,5,2014
2,10003327,20140312,20140312,1,None,1,NaN,CN,CN,53,801,1,Respiratory failure | Listeriosis | Pyrexia | ...,PREDNISONE | DEXAMETHASONE | METHOTREXATE,1 | 1 | 1,8,3,2014
3,10003344,20140312,20150812,2,None,NaN,NaN,US,US,69,801,2,Headache,LETAIRIS | LETAIRIS,1 | 1,1,2,2014
4,10003375,20140312,20140312,1,None,1,NaN,CN,CN,53,801,2,Loss of consciousness | Listeriosis | Pyrexia ...,PREDNISONE | CYCLOPHOSPHAMIDE,1 | 1,6,2,2014


In [11]:
def inspect_raw_report(reaction_term: str = "headache"):
    """
    Fetches one raw report and prints its top-level fields.
    This helps compare the API documentation with the real JSON response.
    """

    search_query = f'patient.reaction.reactionmeddrapt:"{reaction_term}"'

    data = call_openfda(
        search=search_query,
        limit=1,
    )

    report = data["results"][0]

    print("\nTop-level raw fields in one report:")
    for key in sorted(report.keys()):
        print("-", key)

    print("\nPatient-level fields:")
    patient = report.get("patient", {})
    for key in sorted(patient.keys()):
        print("-", f"patient.{key}")

    if "reaction" in patient and patient["reaction"]:
        print("\nReaction-level fields:")
        for key in sorted(patient["reaction"][0].keys()):
            print("-", f"patient.reaction.{key}")

    if "drug" in patient and patient["drug"]:
        print("\nDrug-level fields:")
        for key in sorted(patient["drug"][0].keys()):
            print("-", f"patient.drug.{key}")

    return report



def collect_json_paths(obj, prefix=""):
    """
    Recursively collects all paths inside a nested JSON object.
    Useful for discovering fields inside openFDA records.
    """

    paths = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            new_prefix = f"{prefix}.{key}" if prefix else key
            paths.append(new_prefix)
            paths.extend(collect_json_paths(value, new_prefix))

    elif isinstance(obj, list):
        for item in obj[:1]:
            paths.extend(collect_json_paths(item, f"{prefix}[]"))

    return paths

raw_report = inspect_raw_report("headache")

all_paths = collect_json_paths(raw_report)

print("\nAll discovered JSON paths:")
for path in sorted(set(all_paths)):
    print(path)


Top-level raw fields in one report:
- companynumb
- duplicate
- fulfillexpeditecriteria
- patient
- primarysource
- primarysourcecountry
- receiptdate
- receiptdateformat
- receivedate
- receivedateformat
- receiver
- reportduplicate
- reporttype
- safetyreportid
- safetyreportversion
- sender
- serious
- seriousnessdisabling
- transmissiondate
- transmissiondateformat

Patient-level fields:
- patient.drug
- patient.patientonsetage
- patient.patientonsetageunit
- patient.patientsex
- patient.reaction

Reaction-level fields:
- patient.reaction.reactionmeddrapt
- patient.reaction.reactionmeddraversionpt

Drug-level fields:
- patient.drug.drugadministrationroute
- patient.drug.drugauthorizationnumb
- patient.drug.drugbatchnumb
- patient.drug.drugcharacterization
- patient.drug.drugdosagetext
- patient.drug.drugindication
- patient.drug.drugstartdate
- patient.drug.drugstartdateformat
- patient.drug.drugstructuredosagenumb
- patient.drug.drugstructuredosageunit
- patient.drug.medicinalpro